# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
import os

# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"

# get the data only if it is not already on disk (~110 MB)
if not os.path.exists(jan_2019_trip_data):
    response = requests.get(download_url)
    if response.status_code == 200:
        with open(jan_2019_trip_data, "wb") as f:
            f.write(response.content)
print(jan_2019_trip_data, os.path.getsize(jan_2019_trip_data), "bytes")

yellow_tripdata_2019-01.parquet 110439634 bytes


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

### Part 1: my answers

I chose to do the main part of the lab with the **PySpark DataFrame API**. Part 3 will redo some questions in SQL.

#### Add a column that creates a unique key to identify each record in order to answer questions about individual trips

In [5]:
from pyspark.sql import functions as F, Window

trips = (
    df_trips
    # unique key for each trip
    .withColumn("trip_id", F.monotonically_increasing_id())
    # a few extra columns that I will need for the next questions
    .withColumn(
        "duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime")
         - F.unix_timestamp("tpep_pickup_datetime")) / 60,
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow", F.date_format("tpep_pickup_datetime", "EEEE"))
    .cache()
)

print(f"{trips.count():,} trips")
trips.select("trip_id", "tpep_pickup_datetime", "duration_min",
             "passenger_count", "trip_distance", "total_amount").show(5)

7,696,617 trips


+-------+--------------------+------------------+---------------+-------------+------------+
|trip_id|tpep_pickup_datetime|      duration_min|passenger_count|trip_distance|total_amount|
+-------+--------------------+------------------+---------------+-------------+------------+
|      0| 2019-01-01 00:46:40| 6.666666666666667|            1.0|          1.5|        9.95|
|      1| 2019-01-01 00:59:47|              19.2|            1.0|          2.6|        16.3|
|      2| 2018-12-21 13:48:30| 4.166666666666667|            3.0|          0.0|         5.8|
|      3| 2018-11-28 15:52:25|3.3333333333333335|            5.0|          0.0|        7.55|
|      4| 2018-11-28 15:56:57|               1.6|            5.0|          0.0|       55.55|
+-------+--------------------+------------------+---------------+-------------+------------+
only showing top 5 rows


I used `monotonically_increasing_id()` to create the `trip_id` column. The ids are unique, but they are not always consecutive (each partition gets its own range of numbers). I also call `.cache()`: this way Spark keeps the DataFrame in memory, the next questions run faster, and the ids do not change if the DataFrame is recomputed.

I also added the trip duration in minutes, the pick-up date, hour and day of the week, because I need them for the questions below.

**Checking the data before answering.** When I looked at the data, I saw that some values made no sense (dates in 2001 or 2088, negative durations, a fare of more than 600,000 $). So before answering the questions, I counted these problems:

In [6]:
quality_checks = {
    "pickup_outside_jan_2019": (F.col("tpep_pickup_datetime") < "2019-01-01")
                               | (F.col("tpep_pickup_datetime") >= "2019-02-01"),
    "duration_<=_0": F.col("duration_min") <= 0,
    "duration_>_6h": F.col("duration_min") > 360,
    "distance_<=_0": F.col("trip_distance") <= 0,
    "distance_>_100mi": F.col("trip_distance") > 100,
    "fare_<=_0": F.col("fare_amount") <= 0,
    "passengers_0": F.col("passenger_count") == 0,
    "passengers_null": F.col("passenger_count").isNull(),
}
trips.select(
    [F.sum(cond.cast("int")).alias(name) for name, cond in quality_checks.items()]
).show(vertical=True)

-RECORD 0-------------------------
 pickup_outside_jan_2019 | 537    
 duration_<=_0           | 6557   
 duration_>_6h           | 20534  
 distance_<=_0           | 55089  
 distance_>_100mi        | 32     
 fare_<=_0               | 9770   
 passengers_0            | 117381 
 passengers_null         | 28672  



If I keep these rows, the questions like "longest trip" would just give me the errors. So I created a second DataFrame, `trips_clean`, where I only keep the trips that start in January 2019, last between 0 and 6 hours, are between 0 and 100 miles long, and have a positive fare. I did not filter the passenger count because a 0 there is probably just the driver forgetting to type it.

In [7]:
trips_clean = trips.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01")
    & (F.col("tpep_pickup_datetime") < "2019-02-01")
    & (F.col("duration_min") > 0) & (F.col("duration_min") <= 360)
    & (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100)
    & (F.col("fare_amount") > 0)
)

n_raw, n_clean = trips.count(), trips_clean.count()
print(f"kept {n_clean:,} of {n_raw:,} trips "
      f"({(n_raw - n_clean) / n_raw:.2%} removed)")

kept 7,613,478 of 7,696,617 trips (1.08% removed)


The cleaning only removes about 1 % of the trips, so it does not change the big picture, but it removes the absurd values.

#### Which trip has the highest passanger count

In [8]:
max_passengers = trips.agg(F.max("passenger_count")).first()[0]
print("max passenger_count:", max_passengers)
print("trips with that count:",
      trips.filter(F.col("passenger_count") == max_passengers).count())

(trips
 .filter(F.col("passenger_count") == max_passengers)
 .select("trip_id", "tpep_pickup_datetime", "passenger_count",
         "trip_distance", "duration_min", "total_amount")
 .show())

max passenger_count: 9.0


trips with that count: 9


+-------+--------------------+---------------+-------------+-------------------+------------+
|trip_id|tpep_pickup_datetime|passenger_count|trip_distance|       duration_min|total_amount|
+-------+--------------------+---------------+-------------+-------------------+------------+
| 949956| 2019-01-05 13:12:29|            9.0|          0.0|               0.05|        12.6|
|1296287| 2019-01-07 03:19:36|            9.0|          0.0| 0.4166666666666667|         9.3|
|2012098| 2019-01-10 00:43:10|            9.0|          0.0|0.06666666666666667|        11.3|
|2883995| 2019-01-13 04:13:24|            9.0|          0.0| 1.1666666666666667|       12.25|
|4534707| 2019-01-19 16:45:25|            9.0|          0.0|0.03333333333333333|      110.76|
|4852225| 2019-01-21 03:46:51|            9.0|          0.0|0.08333333333333333|       12.74|
|4997790| 2019-01-21 19:20:28|            9.0|          0.0|               0.05|         9.8|
|7286683| 2019-01-30 18:34:12|            9.0|          0.0|

The highest passenger count is **9**, but there is not only one trip: **9 trips** have 9 passengers. When I look at them, 8 of the 9 have a distance of 0 and last about one minute or less, so I think it is a typing mistake from the driver (a normal yellow cab only takes 4 or 5 passengers). Only one of them (13.38 miles, 27 minutes) looks like a real trip.

#### What is the Average passanger count

In [9]:
trips.groupBy("passenger_count").count().orderBy("passenger_count").show()

trips.select(
    F.round(F.avg("passenger_count"), 3).alias("avg_all_rows"),
    F.round(F.avg(F.when(F.col("passenger_count") > 0,
                         F.col("passenger_count"))), 3).alias("avg_excluding_0"),
).show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL|  28672|
|            0.0| 117381|
|            1.0|5456515|
|            2.0|1113894|
|            3.0| 314692|
|            4.0| 140753|
|            5.0| 323842|
|            6.0| 200811|
|            7.0|     19|
|            8.0|     29|
|            9.0|      9|
+---------------+-------+



+------------+---------------+
|avg_all_rows|avg_excluding_0|
+------------+---------------+
|       1.567|          1.591|
+------------+---------------+



The average passenger count is **about 1.57**. If I remove the trips with 0 passengers (which are errors), it becomes 1.59, so it does not change much. The table also shows that most trips (about 71 %) have only one passenger.

#### Shortest/longest trip by distance? by time?.

In [10]:
cols = ["trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "trip_distance", "duration_min", "fare_amount"]

print("Shortest by distance")
trips_clean.orderBy("trip_distance").select(cols).show(3)
print("Longest by distance")
trips_clean.orderBy(F.desc("trip_distance")).select(cols).show(3)
print("Shortest by time")
trips_clean.orderBy("duration_min").select(cols).show(3)
print("Longest by time")
trips_clean.orderBy(F.desc("duration_min")).select(cols).show(3)

Shortest by distance


+-------+--------------------+---------------------+-------------+-------------------+-----------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|       duration_min|fare_amount|
+-------+--------------------+---------------------+-------------+-------------------+-----------+
|  12564| 2019-01-01 00:02:54|  2019-01-01 00:02:58|         0.01|0.06666666666666667|        2.5|
|  18829| 2019-01-01 01:03:20|  2019-01-01 01:04:01|         0.01| 0.6833333333333333|        2.5|
|  14319| 2019-01-01 01:34:48|  2019-01-01 01:35:16|         0.01| 0.4666666666666667|        2.5|
+-------+--------------------+---------------------+-------------+-------------------+-----------+
only showing top 3 rows
Longest by distance


+-------+--------------------+---------------------+-------------+------------------+-----------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|fare_amount|
+-------+--------------------+---------------------+-------------+------------------+-----------+
|3001530| 2019-01-13 17:40:12|  2019-01-13 20:02:32|        98.38|142.33333333333334|      300.0|
|4663116| 2019-01-20 03:44:56|  2019-01-20 05:18:54|        96.13| 93.96666666666667|      200.0|
|4223591| 2019-01-18 11:34:58|  2019-01-18 13:10:34|        91.86|              95.6|      240.0|
+-------+--------------------+---------------------+-------------+------------------+-----------+
only showing top 3 rows
Shortest by time


+-------+--------------------+---------------------+-------------+--------------------+-----------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|        duration_min|fare_amount|
+-------+--------------------+---------------------+-------------+--------------------+-----------+
|  52175| 2019-01-01 03:31:50|  2019-01-01 03:31:51|         1.65|0.016666666666666666|        7.5|
| 100141| 2019-01-01 13:34:26|  2019-01-01 13:34:27|         1.24|0.016666666666666666|        6.5|
|  76842| 2019-01-01 10:19:45|  2019-01-01 10:19:46|         0.01|0.016666666666666666|        2.5|
+-------+--------------------+---------------------+-------------+--------------------+-----------+
only showing top 3 rows
Longest by time


+-------+--------------------+---------------------+-------------+------------------+-----------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|fare_amount|
+-------+--------------------+---------------------+-------------+------------------+-----------+
| 778929| 2019-01-04 18:00:34|  2019-01-05 00:00:00|         0.55|359.43333333333334|        4.5|
|5208216| 2019-01-22 18:01:50|  2019-01-23 00:00:00|         1.26| 358.1666666666667|        7.5|
| 544150| 2019-01-03 18:01:59|  2019-01-04 00:00:00|         0.56|358.01666666666665|        5.0|
+-------+--------------------+---------------------+-------------+------------------+-----------+
only showing top 3 rows


In [ ]:
# same question on the RAW data, to compare with the cleaned data
print("Longest by time (raw data, before cleaning)")
trips.orderBy(F.desc("duration_min")).select(cols).show(3)
print("Longest by distance (raw data, before cleaning)")
trips.orderBy(F.desc("trip_distance")).select(cols).show(3)

Longest by time (raw data, before cleaning)
+-------+--------------------+---------------------+-------------+------------------+-----------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|fare_amount|
+-------+--------------------+---------------------+-------------+------------------+-----------+
|  68267| 2019-01-01 07:01:20|  2019-01-31 14:29:21|          1.2| 43648.01666666667|        6.5|
| 592262| 2019-01-03 22:24:36|  2019-01-27 10:41:17|          1.1|33856.683333333334|        8.5|
| 875856| 2019-01-05 04:21:40|  2019-01-27 01:53:46|          3.9|           31532.1|       16.5|
+-------+--------------------+---------------------+-------------+------------------+-----------+
only showing top 3 rows
Longest by distance (raw data, before cleaning)
+-------+--------------------+---------------------+-------------+-----------------+-----------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|     duration_min|fare_amount|
+---

**On the raw data** (second table), the longest trip by time is the trip **68267**: it starts on 1 January 2019 at 07:01 and ends on **31 January** at 14:29, so about **30 days** (43,648 minutes), for only 1.2 miles and a fare of 6.50 $. It is obviously not a real trip: the meter was never closed. The same for distance: the longest raw trip is 831.8 miles in 9 minutes, which is impossible. This is why I answer on the cleaned data.

**On the cleaned data** (first table):

- **Longest by distance**: 98.38 miles in about 2 h 20, for a fare of 300 $. It looks like a real trip outside of the city.
- **Shortest by distance**: 0.01 mile. These are probably trips where the meter was started and stopped right away (cancelled rides).
- **Shortest by time**: 1 second, same explanation.
- **Longest by time**: almost 6 hours (my limit). The longest ones all end at exactly `00:00:00`, which is strange. I checked: 218 of the 327 trips longer than 3 hours end exactly at midnight, so I think the meter is closed automatically at midnight and these are not real trips.

So even after cleaning, the min and max values are still a bit suspicious.

#### busiest day/slowest single day

In [11]:
trips_per_day = trips_clean.groupBy("pickup_date").count()

print("Busiest days")
trips_per_day.orderBy(F.desc("count")).show(3)
print("Slowest days")
trips_per_day.orderBy("count").show(3)

Busiest days


+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|289511|
| 2019-01-11|288591|
| 2019-01-17|281692|
+-----------+------+
only showing top 3 rows
Slowest days


+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-01|186690|
| 2019-01-21|190356|
| 2019-01-02|196301|
+-----------+------+
only showing top 3 rows


- **Busiest day**: Friday **25 January 2019**, with about 290,000 trips.
- **Slowest day**: **1 January 2019** (New Year's Day), with about 187,000 trips. The second slowest day is 21 January, which is Martin Luther King Day, also a holiday in the US. It makes sense that people take fewer taxis on holidays.

#### busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )

In [12]:
# by hour
trips_clean.groupBy("pickup_hour").count().orderBy("pickup_hour").show(24)

# by time of day
time_of_day = (
    F.when(F.col("pickup_hour").between(6, 11), "1. morning (6-11h)")
    .when(F.col("pickup_hour").between(12, 16), "2. afternoon (12-16h)")
    .when(F.col("pickup_hour").between(17, 21), "3. evening (17-21h)")
    .otherwise("4. night (22-5h)")
)
(trips_clean
 .groupBy(time_of_day.alias("time_of_day"))
 .agg(F.count("*").alias("trips"))
 .orderBy("time_of_day")
 .show(truncate=False))

# keep the bucket as a column, it is reused in part 2
trips_clean = trips_clean.withColumn("time_of_day_bucket", time_of_day)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0|204926|
|          1|147061|
|          2|107553|
|          3| 76491|
|          4| 59865|
|          5| 73922|
|          6|175848|
|          7|301151|
|          8|370187|
|          9|362456|
|         10|357852|
|         11|371671|
|         12|397109|
|         13|399846|
|         14|428380|
|         15|447528|
|         16|415820|
|         17|463677|
|         18|510847|
|         19|471242|
|         20|419514|
|         21|406197|
|         22|365587|
|         23|278748|
+-----------+------+



+---------------------+-------+
|time_of_day          |trips  |
+---------------------+-------+
|1. morning (6-11h)   |1939165|
|2. afternoon (12-16h)|2088683|
|3. evening (17-21h)  |2271477|
|4. night (22-5h)     |1314153|
+---------------------+-------+



- By hour, the **busiest hour is 18h** (about 511,000 trips in the month), which is when people leave work. The **slowest hour is 4h** (about 60,000 trips).
- With my buckets, the **evening (17-21h)** is the busiest and the **night (22-5h)** is the slowest, even if the night bucket is the longest one (8 hours).

#### On average which day of the week is slowest/busiest

In [13]:
(trips_per_day
 .withColumn("day_of_week", F.date_format("pickup_date", "EEEE"))
 .groupBy("day_of_week")
 .agg(F.count("*").alias("days_in_month"),
      F.sum("count").alias("total_trips"),
      F.round(F.avg("count")).alias("avg_trips_per_day"))
 .orderBy(F.desc("avg_trips_per_day"))
 .show())

+-----------+-------------+-----------+-----------------+
|day_of_week|days_in_month|total_trips|avg_trips_per_day|
+-----------+-------------+-----------+-----------------+
|     Friday|            4|    1075595|         268899.0|
|   Thursday|            5|    1342909|         268582.0|
|  Wednesday|            5|    1252125|         250425.0|
|   Saturday|            4|     999135|         249784.0|
|    Tuesday|            5|    1195774|         239155.0|
|     Monday|            4|     898052|         224513.0|
|     Sunday|            4|     849888|         212472.0|
+-----------+-------------+-----------+-----------------+



In January 2019, there are 5 Tuesdays, Wednesdays and Thursdays, but only 4 of the other days. If I just sum the trips, these days are advantaged. That is why I first counted the trips per date, and then I computed the **average** per day of the week.

- **Busiest day on average: Friday** (about 269,000 trips per day), just before Thursday.
- **Slowest day on average: Sunday** (about 212,000 trips per day).

If I had used the total instead of the average, Thursday would have been first only because there are 5 Thursdays in the month.

#### Does trip distance or num passangers affect tip amount

In [14]:
trips_clean.groupBy("payment_type").agg(
    F.count("*").alias("trips"),
    F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
).orderBy("payment_type").show()

+------------+-------+-------+
|payment_type|  trips|avg_tip|
+------------+-------+-------+
|           0|  28106|   0.01|
|           1|5448292|   2.53|
|           2|2105568|    0.0|
|           3|  24015|    0.0|
|           4|   7497|   0.01|
+------------+-------+-------+



I noticed that the tip is almost always 0 when the payment type is not 1 (credit card). It is because cash tips are not recorded in the data. So I only kept credit card trips to study the tips, otherwise the cash trips would make the tips look smaller than they are.

In [15]:
card_trips = (
    trips_clean
    .filter(F.col("payment_type") == 1)
    .withColumn("tip_pct", F.col("tip_amount") / F.col("fare_amount") * 100)
)

card_trips.select(
    F.round(F.corr("trip_distance", "tip_amount"), 3).alias("corr_distance_tip"),
    F.round(F.corr("passenger_count", "tip_amount"), 3).alias("corr_passengers_tip"),
    F.round(F.corr("trip_distance", "tip_pct"), 3).alias("corr_distance_tip_pct"),
).show()

distance_bucket = (
    F.when(F.col("trip_distance") < 1, "1. < 1 mi")
    .when(F.col("trip_distance") < 3, "2. 1-3 mi")
    .when(F.col("trip_distance") < 5, "3. 3-5 mi")
    .when(F.col("trip_distance") < 10, "4. 5-10 mi")
    .otherwise("5. 10+ mi")
)
print("Tip by distance")
(card_trips
 .groupBy(distance_bucket.alias("distance"))
 .agg(F.count("*").alias("trips"),
      F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
      F.round(F.percentile_approx("tip_pct", 0.5), 1).alias("median_tip_pct"))
 .orderBy("distance")
 .show())

print("Tip by passenger count")
(card_trips
 .groupBy("passenger_count")
 .agg(F.count("*").alias("trips"),
      F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
      F.round(F.percentile_approx("tip_pct", 0.5), 1).alias("median_tip_pct"))
 .orderBy("passenger_count")
 .show())

+-----------------+-------------------+---------------------+
|corr_distance_tip|corr_passengers_tip|corr_distance_tip_pct|
+-----------------+-------------------+---------------------+
|            0.711|              0.011|               -0.007|
+-----------------+-------------------+---------------------+

Tip by distance


+----------+-------+-------+--------------+
|  distance|  trips|avg_tip|median_tip_pct|
+----------+-------+-------+--------------+
| 1. < 1 mi|1371841|   1.34|          23.7|
| 2. 1-3 mi|2739285|   1.98|          22.1|
| 3. 3-5 mi| 579730|   3.08|          21.2|
|4. 5-10 mi| 432168|   4.69|          20.9|
| 5. 10+ mi| 325268|   8.39|          20.8|
+----------+-------+-------+--------------+

Tip by passenger count


+---------------+-------+-------+--------------+
|passenger_count|  trips|avg_tip|median_tip_pct|
+---------------+-------+-------+--------------+
|            0.0|  82495|   2.49|          22.1|
|            1.0|3909526|   2.51|          22.2|
|            2.0| 776104|   2.59|          22.3|
|            3.0| 217100|   2.57|          22.3|
|            4.0|  91344|   2.57|          22.3|
|            5.0| 229675|    2.6|          22.3|
|            6.0| 142034|    2.6|          22.3|
|            7.0|      4|  13.93|           6.7|
|            8.0|      9|  12.17|          20.2|
|            9.0|      1|    0.0|           0.0|
+---------------+-------+-------+--------------+



- **Distance: yes.** The correlation between the distance and the tip is **0.71**, which is quite strong. The average tip goes from 1.34 $ for trips under 1 mile to 8.39 $ for trips over 10 miles. But when I look at the tip as a **percentage** of the fare, the correlation is almost 0: people give about 21 to 24 % of the fare, whatever the distance. So the tip is bigger on long trips only because the fare is bigger.
- **Number of passengers: no.** The correlation is 0.01, and the average tip stays around 2.50 $ from 0 to 6 passengers. The lines with 7, 8 and 9 passengers have fewer than 10 trips, so I do not take them into account.

#### What was the highest "extra" charge and which trip

In [16]:
extra_cols = ["trip_id", "tpep_pickup_datetime", "extra", "fare_amount",
              "total_amount", "trip_distance", "duration_min", "RatecodeID"]

print("Raw data")
trips.orderBy(F.desc("extra")).select(extra_cols).show(3)
print("Cleaned data")
trips_clean.orderBy(F.desc("extra")).select(extra_cols).show(3)
print("Most common extra values")
trips.groupBy("extra").count().orderBy(F.desc("count")).show(8)

Raw data


+-------+--------------------+------+-----------+------------+-------------+-----------------+----------+
|trip_id|tpep_pickup_datetime| extra|fare_amount|total_amount|trip_distance|     duration_min|RatecodeID|
+-------+--------------------+------+-----------+------------+-------------+-----------------+----------+
|5323483| 2019-01-23 08:58:09|535.38|  355676.98|   356214.78|          0.0|              0.0|       1.0|
|7453230| 2019-01-31 10:06:09| 23.04|        4.5|       28.34|          0.0|              0.0|       1.0|
| 543203| 2019-01-03 18:32:36|  18.5|       61.0|        92.3|         16.6|72.88333333333334|       3.0|
+-------+--------------------+------+-----------+------------+-------------+-----------------+----------+
only showing top 3 rows
Cleaned data


+-------+--------------------+-----+-----------+------------+-------------+-----------------+----------+
|trip_id|tpep_pickup_datetime|extra|fare_amount|total_amount|trip_distance|     duration_min|RatecodeID|
+-------+--------------------+-----+-----------+------------+-------------+-----------------+----------+
| 543203| 2019-01-03 18:32:36| 18.5|       61.0|        92.3|         16.6|72.88333333333334|       3.0|
|4017135| 2019-01-17 16:24:12| 18.5|       91.5|      134.32|        34.39|65.63333333333334|       3.0|
| 548308| 2019-01-03 18:19:33| 18.5|       47.5|       94.56|        16.74|            36.35|       3.0|
+-------+--------------------+-----+-----------+------------+-------------+-----------------+----------+
only showing top 3 rows
Most common extra values


+-----+-------+
|extra|  count|
+-----+-------+
|  0.0|4200521|
|  0.5|2116601|
|  1.0|1316575|
|  4.5|  31240|
| 2.75|  26583|
| -0.5|   2201|
|  5.5|   1377|
| -1.0|    863|
+-----+-------+
only showing top 8 rows


- In the raw data, the highest extra is **535.38 $**, on the trip of 23 January 2019 at 08:58:09 (its `trip_id` is in the table above). But this trip has a distance of 0, a duration of 0 and a fare of 355,676.98 $, so it is clearly an error.
- In the cleaned data, the highest extra is **18.50 $**. These trips have `RatecodeID = 3`, which means they go to Newark airport, so the extra fees make sense.
- Normally, the extra is 0 $, 0.50 $ or 1 $ (these are the evening and rush hour surcharges): these 3 values represent more than 99 % of the trips.

#### Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [17]:
# pick-up dates outside January 2019, by year and month
(trips
 .filter((F.year("tpep_pickup_datetime") != 2019)
         | (F.month("tpep_pickup_datetime") != 1))
 .groupBy(F.year("tpep_pickup_datetime").alias("year"),
          F.month("tpep_pickup_datetime").alias("month"))
 .count()
 .orderBy("year", "month")
 .show(30))

# the biggest fares
trips.orderBy(F.desc("fare_amount")).select(
    "trip_id", "tpep_pickup_datetime", "trip_distance",
    "duration_min", "fare_amount").show(3)

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2001|    2|    1|
|2003|    1|    2|
|2008|   12|   22|
|2009|    1|   50|
|2018|   11|   11|
|2018|   12|  355|
|2019|    2|   72|
|2019|    3|    5|
|2019|    4|    6|
|2019|    5|    1|
|2019|    6|    2|
|2019|    7|    6|
|2019|    8|    1|
|2019|    9|    1|
|2088|    1|    2|
+----+-----+-----+



+-------+--------------------+-------------+------------+-----------+
|trip_id|tpep_pickup_datetime|trip_distance|duration_min|fare_amount|
+-------+--------------------+-------------+------------+-----------+
|2499655| 2019-01-11 19:33:15|          2.4|        19.9|  623259.86|
|5323483| 2019-01-23 08:58:09|          0.0|         0.0|  355676.98|
|2159971| 2019-01-10 16:08:10|          0.0|         0.0|    36090.3|
+-------+--------------------+-------------+------------+-----------+
only showing top 3 rows


Yes, there are a lot of strange data points. Here is a summary of what I found during this part (the numbers come from the checks at the beginning):

| problem | number of rows | why I think it is an outlier |
|---|---|---|
| pick-up outside January 2019 | 537 | the file should only contain January 2019, but some dates are in 2001, 2008 or even 2088, so the clock of the meter was wrong |
| duration <= 0 | 6,557 | the drop-off is before (or at the same time as) the pick-up, which is impossible |
| duration > 6 h | 20,534 | nobody takes a taxi for 6 hours in the city, the meter was probably left on (many of them end exactly at midnight) |
| distance = 0 | 55,089 | probably cancelled trips |
| distance > 100 miles | 32 | for example 831.8 miles in 9 minutes, which would be more than 5,000 mph |
| fare <= 0 | 9,770 (7,129 negative) | a negative fare is probably a refund or a dispute, not a real trip |
| passengers = 0 | 117,381 | the driver did not type the number of passengers |
| very high fares | a few | for example 623,259.86 $ for a 2.4 mile trip |

Even in my cleaned data, there are still about 5,900 trips with an average speed above 80 mph, which is not possible in New York. I could add another filter on the speed to go further.

What I learned from this part: before answering questions with the min or the max of a column, I have to check the data, because the extreme values are very often errors.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

### Part 2: my answers

#### Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions

In [18]:
# get the taxi zone lookup table (only if not already on disk)
import os
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_file = "taxi_zone_lookup.csv"

if not os.path.exists(zone_file):
    response = requests.get(zone_url)
    if response.status_code == 200:
        with open(zone_file, "wb") as f:
            f.write(response.content)

# csv is not self-describing: we need the header and a schema
# (inferSchema is fine here, the file only has ~265 rows)
df_zones = spark.read.csv(zone_file, header=True, inferSchema=True)
df_zones.printSchema()
df_zones.show(5)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


I loaded the zone lookup table with the same logic as the 2019 trips (download only if the file is not there). The difference is that it is a **csv** file, so Spark needs `header=True` to use the first line as column names and `inferSchema=True` to guess the types. The table is small (265 zones), and each zone has a `LocationID` and a `Borough`.

The trips only have `PULocationID` and `DOLocationID` (pick-up and drop-off zone), so I need to **join** the two tables to know the borough of each trip:

In [19]:
# add the pick-up and drop-off borough to each trip with two joins
# (the zone table is tiny, so broadcast() sends a copy of it to every
# executor instead of shuffling the 7.6 million trips)
pickup_zones = df_zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
)
dropoff_zones = df_zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
)

trips_boro = (
    trips_clean
    .join(F.broadcast(pickup_zones), on="PULocationID", how="left")
    .join(F.broadcast(dropoff_zones), on="DOLocationID", how="left")
    .cache()
)
print(f"{trips_boro.count():,} trips with their boroughs")
trips_boro.select("trip_id", "PULocationID", "pickup_borough",
                  "DOLocationID", "dropoff_borough").show(5)

7,613,478 trips with their boroughs


+-------+------------+--------------+------------+---------------+
|trip_id|PULocationID|pickup_borough|DOLocationID|dropoff_borough|
+-------+------------+--------------+------------+---------------+
|      0|         151|     Manhattan|         239|      Manhattan|
|      1|         239|     Manhattan|         246|      Manhattan|
|      7|         163|     Manhattan|         229|      Manhattan|
|      8|         229|     Manhattan|           7|         Queens|
|      9|         141|     Manhattan|         234|      Manhattan|
+-------+------------+--------------+------------+---------------+
only showing top 5 rows


I did two joins: one on `PULocationID` to get the pick-up borough, and one on `DOLocationID` to get the drop-off borough. I used a `left` join so that no trip is lost if a zone is missing. I also used `broadcast()` because the zone table is very small: Spark sends a copy of it to every worker, so it does not need to move the 7.6 million trips around. I kept working with the cleaned trips from part 1.

#### which borough had most pickups? dropoffs?

In [20]:
pickups = trips_boro.groupBy(F.col("pickup_borough").alias("borough")) \
    .agg(F.count("*").alias("pickups"))
dropoffs = trips_boro.groupBy(F.col("dropoff_borough").alias("borough")) \
    .agg(F.count("*").alias("dropoffs"))

(pickups
 .join(dropoffs, on="borough", how="outer")
 .withColumn("pickups_pct",
             F.round(F.col("pickups") / trips_boro.count() * 100, 2))
 .orderBy(F.desc("pickups"))
 .show())

+-------------+-------+--------+-----------+
|      borough|pickups|dropoffs|pickups_pct|
+-------------+-------+--------+-----------+
|    Manhattan|6898060| 6766655|       90.6|
|       Queens| 455302|  325428|       5.98|
|      Unknown| 151157|  140623|       1.99|
|     Brooklyn|  89229|  296966|       1.17|
|        Bronx|  17175|   56853|       0.23|
|          N/A|   2096|   14417|       0.03|
|Staten Island|    322|    2132|        0.0|
|          EWR|    137|   10404|        0.0|
+-------------+-------+--------+-----------+



- **Most pick-ups: Manhattan**, by far, with about **6.9 million** pick-ups, i.e. **90.6 %** of all trips. It makes sense because yellow taxis mostly drive in Manhattan.
- **Most drop-offs: Manhattan** too (6.8 million).
- After Manhattan, Queens is second for both. I think it is because of the two airports (JFK and LaGuardia are in Queens).
- Brooklyn has about 3 times more drop-offs than pick-ups (297,000 vs 89,000): people take a yellow taxi from Manhattan to go home in Brooklyn, but it is harder to find one in Brooklyn to come back.
- `Unknown` and `N/A` are not real boroughs, they are the zones 264 (`Unknown`) and 265 (`N/A`, outside of New York) used when the location is unknown or outside the city.

#### what are the busy/slow times by borough 

In [21]:
# number of pick-ups for each (borough, hour)
per_borough_hour = (
    trips_boro
    .groupBy("pickup_borough", "pickup_hour")
    .agg(F.count("*").alias("trips"))
)

# rank the hours inside each borough, from the busiest and from the slowest
w_busy = Window.partitionBy("pickup_borough").orderBy(F.desc("trips"))
w_slow = Window.partitionBy("pickup_borough").orderBy("trips")

busiest = (per_borough_hour
           .withColumn("r", F.row_number().over(w_busy)).filter("r = 1")
           .select("pickup_borough",
                   F.col("pickup_hour").alias("busiest_hour"),
                   F.col("trips").alias("trips_busiest_hour")))
slowest = (per_borough_hour
           .withColumn("r", F.row_number().over(w_slow)).filter("r = 1")
           .select("pickup_borough",
                   F.col("pickup_hour").alias("slowest_hour"),
                   F.col("trips").alias("trips_slowest_hour")))

busiest.join(slowest, on="pickup_borough").orderBy("pickup_borough").show()

# same thing with the time-of-day buckets from part 1
(trips_boro
 .groupBy("pickup_borough")
 .pivot("time_of_day_bucket")
 .count()
 .orderBy("pickup_borough")
 .show(truncate=False))

+--------------+------------+------------------+------------+------------------+
|pickup_borough|busiest_hour|trips_busiest_hour|slowest_hour|trips_slowest_hour|
+--------------+------------+------------------+------------+------------------+
|         Bronx|           7|              1718|           3|               199|
|      Brooklyn|           8|              6793|           3|              1839|
|           EWR|          15|                21|           4|                 1|
|     Manhattan|          18|            468442|           4|             52604|
|           N/A|          14|               134|           4|                42|
|        Queens|          21|             28846|           3|              2897|
| Staten Island|           8|                34|          22|                 2|
|       Unknown|          18|             10294|           4|              1208|
+--------------+------------+------------------+------------+------------------+



+--------------+------------------+---------------------+-------------------+----------------+
|pickup_borough|1. morning (6-11h)|2. afternoon (12-16h)|3. evening (17-21h)|4. night (22-5h)|
+--------------+------------------+---------------------+-------------------+----------------+
|Bronx         |7389              |4329                 |2776               |2681            |
|Brooklyn      |28582             |18196                |18715              |23736           |
|EWR           |24                |69                   |35                 |9               |
|Manhattan     |1761030           |1897875              |2067026            |1172129         |
|N/A           |583               |521                  |502                |490             |
|Queens        |103456            |124891               |137044             |89911           |
|Staten Island |141               |78                   |59                 |44              |
|Unknown       |37960             |42724          

I used a **window function** (`Window.partitionBy("pickup_borough")`) to rank the hours inside each borough, and I kept the first one (busiest) and the last one (slowest).

- **Manhattan**: busiest at **18h**, slowest at **4h**, like the whole city in part 1 (normal, since Manhattan is 90 % of the trips).
- **Bronx and Brooklyn**: busiest in the **morning** (7h and 8h). People probably take a taxi to go to work in Manhattan.
- **Queens**: busiest at **21h**. It fits with the airports, where there are a lot of flights arriving in the evening.
- In all the boroughs the slowest hours are at night (3h or 4h), except Staten Island, where there are so few trips (322 in the month) that the result is not really meaningful.

The second table (with `pivot`) shows the same thing with the time buckets of part 1: the evening is the busiest bucket in Manhattan and Queens, but the morning is the busiest one in the Bronx and in Brooklyn.

#### what are the busiest days of the week by borough?

In [22]:
# like in part 1: count per date first, then average per day of the week,
# because january 2019 does not have the same number of each weekday
per_borough_day = (
    trips_boro
    .groupBy("pickup_borough", "pickup_date")
    .agg(F.count("*").alias("trips"))
    .withColumn("day_of_week", F.date_format("pickup_date", "EEEE"))
    .groupBy("pickup_borough", "day_of_week")
    .agg(F.round(F.avg("trips")).alias("avg_trips_per_day"))
)

w_day = Window.partitionBy("pickup_borough").orderBy(F.desc("avg_trips_per_day"))
(per_borough_day
 .withColumn("rank", F.row_number().over(w_day))
 .filter("rank <= 2")
 .orderBy("pickup_borough", "rank")
 .show(20))

+--------------+-----------+-----------------+----+
|pickup_borough|day_of_week|avg_trips_per_day|rank|
+--------------+-----------+-----------------+----+
|         Bronx|     Friday|            638.0|   1|
|         Bronx|   Thursday|            593.0|   2|
|      Brooklyn|     Friday|           3182.0|   1|
|      Brooklyn|    Tuesday|           3074.0|   2|
|           EWR|     Friday|              8.0|   1|
|           EWR|     Sunday|              5.0|   2|
|     Manhattan|     Friday|         244361.0|   1|
|     Manhattan|   Thursday|         244113.0|   2|
|           N/A|  Wednesday|             74.0|   1|
|           N/A|    Tuesday|             74.0|   2|
|        Queens|     Monday|          16104.0|   1|
|        Queens|     Friday|          15492.0|   2|
| Staten Island|     Friday|             15.0|   1|
| Staten Island|   Thursday|             11.0|   2|
|       Unknown|   Thursday|           5500.0|   1|
|       Unknown|     Friday|           5146.0|   2|
+-----------

I did the same as in part 1 (average per day instead of total, because the month has 5 Tuesdays, Wednesdays and Thursdays). Then I used a window again to keep the 2 busiest days of each borough.

- **Friday** is the busiest day in most boroughs: Manhattan, Brooklyn, Bronx, Staten Island.
- **Queens** is different: its busiest day is **Monday**. My guess is that it is linked to the airports, with a lot of people coming back from a trip or leaving for a business trip at the start of the week.
- For EWR, N/A and Staten Island, there are less than 20 trips per day, so I would not conclude anything.

#### what is the average trip distance by borough?

In [23]:
(trips_boro
 .groupBy("pickup_borough")
 .agg(F.count("*").alias("trips"),
      F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
      F.round(F.percentile_approx("trip_distance", 0.5), 2).alias("median_distance_mi"))
 .orderBy(F.desc("avg_distance_mi"))
 .show())

+--------------+-------+---------------+------------------+
|pickup_borough|  trips|avg_distance_mi|median_distance_mi|
+--------------+-------+---------------+------------------+
| Staten Island|    322|           13.9|              12.6|
|        Queens| 455302|           11.6|             10.78|
|           EWR|    137|           7.75|               1.3|
|         Bronx|  17175|           7.57|              5.97|
|           N/A|   2096|           5.29|               1.8|
|      Brooklyn|  89229|           4.91|              3.26|
|       Unknown| 151157|           2.54|               1.5|
|     Manhattan|6898060|           2.24|              1.47|
+--------------+-------+---------------+------------------+



The average distance depends a lot on the borough:

- **Manhattan** has the shortest trips: **2.24 miles** on average (1.47 for the median). Manhattan is small and dense, so people take the taxi for short distances.
- **Queens** has much longer trips (**11.6 miles**) because a lot of them start at the airports and go to Manhattan.
- **Staten Island** has the longest trips on average (13.9 miles), but with only 322 trips.

I also added the median because some boroughs have a few very long trips that increase the average (for EWR, the average is 7.75 miles but the median is only 1.3).

#### what is the average trip fare by borough?

In [24]:
(trips_boro
 .groupBy("pickup_borough")
 .agg(F.count("*").alias("trips"),
      F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
      F.round(F.avg("total_amount"), 2).alias("avg_total_paid"))
 .orderBy(F.desc("avg_fare"))
 .show())

+--------------+-------+--------+--------------+
|pickup_borough|  trips|avg_fare|avg_total_paid|
+--------------+-------+--------+--------------+
|           EWR|    137|    77.8|         95.38|
| Staten Island|    322|   44.67|         53.27|
|           N/A|   2096|   40.04|         47.71|
|        Queens| 455302|   35.69|         45.17|
|         Bronx|  17175|   26.89|         29.99|
|      Brooklyn|  89229|   18.83|         21.82|
|       Unknown| 151157|   11.76|         14.92|
|     Manhattan|6898060|   10.72|         13.59|
+--------------+-------+--------+--------------+



The average fare follows the average distance, which is logical because the fare is mostly calculated with the distance:

- **Manhattan** is the cheapest on average: **10.72 $** (13.59 $ in total with taxes, tip and surcharges).
- **Queens**: **35.69 $**, because of the airport trips (JFK to Manhattan has a flat fare of 52 $ in 2019).
- **EWR** (Newark airport) is the most expensive (77.80 $), but there are only 137 pick-ups there.

#### highest/lowest faire amounts for a trip, what burough is associated with the each

In [25]:
fare_cols = ["trip_id", "tpep_pickup_datetime", "pickup_borough",
             "dropoff_borough", "trip_distance", "duration_min",
             "RatecodeID", "fare_amount"]

print("Highest fares")
trips_boro.orderBy(F.desc("fare_amount")).select(fare_cols).show(5)
print("Lowest fares")
trips_boro.orderBy("fare_amount").select(fare_cols).show(5)
print("How many trips have the lowest fare:",
      trips_boro.filter(F.col("fare_amount") ==
                        trips_boro.agg(F.min("fare_amount")).first()[0]).count())

Highest fares


+-------+--------------------+--------------+---------------+-------------+------------------+----------+-----------+
|trip_id|tpep_pickup_datetime|pickup_borough|dropoff_borough|trip_distance|      duration_min|RatecodeID|fare_amount|
+-------+--------------------+--------------+---------------+-------------+------------------+----------+-----------+
|2499655| 2019-01-11 19:33:15|     Manhattan|      Manhattan|          2.4|              19.9|       1.0|  623259.86|
| 478819| 2019-01-03 13:08:33|       Unknown|        Unknown|          0.1|0.3333333333333333|       6.0|    6666.65|
|6827249| 2019-01-28 21:31:15|       Unknown|        Unknown|          1.0|               2.8|       1.0|     3004.0|
|5507759| 2019-01-23 20:01:05|     Manhattan|      Manhattan|          0.1|0.7666666666666667|       5.0|      684.0|
|1037944| 2019-01-05 20:30:17|        Queens|         Queens|          1.5|1.0333333333333334|       5.0|      500.0|
+-------+--------------------+--------------+-----------

+-------+--------------------+--------------+---------------+-------------+------------------+----------+-----------+
|trip_id|tpep_pickup_datetime|pickup_borough|dropoff_borough|trip_distance|      duration_min|RatecodeID|fare_amount|
+-------+--------------------+--------------+---------------+-------------+------------------+----------+-----------+
|  35276| 2019-01-01 02:50:06|     Manhattan|            N/A|          8.2|52.266666666666666|       5.0|       0.01|
|  10267| 2019-01-01 00:53:53|     Manhattan|            N/A|         15.1|27.516666666666666|       5.0|       0.01|
| 178724| 2019-01-01 21:57:08|     Manhattan|      Manhattan|          0.1|               0.6|       5.0|       0.01|
|  43396| 2019-01-01 03:05:36|     Manhattan|      Manhattan|          0.5| 5.083333333333333|       5.0|       0.01|
|  21537| 2019-01-01 01:54:14|     Manhattan|      Manhattan|          0.1| 3.533333333333333|       5.0|       0.01|
+-------+--------------------+--------------+-----------

How many trips have the lowest fare: 304


- **Highest fare**: **623,259.86 $**, for a trip from **Manhattan to Manhattan** of 2.4 miles and 20 minutes. It is clearly an error (it is the same outlier as in part 1). My cleaning in part 1 did not remove it because I did not put a maximum on the fare. The next ones (6,666.65 $ and 3,004 $) are in the `Unknown` borough and last less than 3 minutes, so they are errors too. The first fare that looks real is 500 $ for a trip in Queens with `RatecodeID = 5` (negotiated fare).
- **Lowest fare**: **0.01 $**, for **304 trips**. The ones in the table all start in **Manhattan** and have `RatecodeID = 5` (negotiated fare), so I think the driver typed a symbolic price, for example for a trip that was paid in another way.

So for this question, the extreme values do not really say anything about the boroughs: they show that the fare column also has typing errors.

#### load the dataset from the most recently available january, is there a change to any of the average metrics.

In [26]:
# most recent january: try 2026 first, fall back to 2025 if not published
for year in (2026, 2025):
    recent_file = f"yellow_tripdata_{year}-01.parquet"
    recent_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{recent_file}"
    if os.path.exists(recent_file):
        break
    response = requests.get(recent_url)
    if response.status_code == 200:
        with open(recent_file, "wb") as f:
            f.write(response.content)
        break
print("using", recent_file)

df_trips_recent = spark.read.parquet(recent_file)

using yellow_tripdata_2026-01.parquet


The most recent january available is **January 2026**, so I loaded `yellow_tripdata_2026-01.parquet` with the same code. To compare the two years in a fair way, I applied the **same cleaning** as for 2019 (with the dates of January 2026).

In [27]:
# same derived columns and same cleaning as for 2019, but for january 2026
recent_clean = (
    df_trips_recent
    .withColumn(
        "duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime")
         - F.unix_timestamp("tpep_pickup_datetime")) / 60,
    )
    .filter(
        (F.col("tpep_pickup_datetime") >= "2026-01-01")
        & (F.col("tpep_pickup_datetime") < "2026-02-01")
        & (F.col("duration_min") > 0) & (F.col("duration_min") <= 360)
        & (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100)
        & (F.col("fare_amount") > 0)
    )
    .cache()
)
print(f"2026: kept {recent_clean.count():,} of {df_trips_recent.count():,} trips")

2026: kept 3,515,654 of 3,724,889 trips


In [28]:
def average_metrics(df, label):
    """Main average metrics of a cleaned trips DataFrame."""
    card = df.filter(F.col("payment_type") == 1)
    main = df.agg(
        F.count("*").alias("trips"),
        F.avg("passenger_count").alias("avg_passengers"),
        F.avg("trip_distance").alias("avg_distance_mi"),
        F.avg("duration_min").alias("avg_duration_min"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("total_amount").alias("avg_total_paid"),
        (F.sum((F.col("payment_type") == 1).cast("int"))
         / F.count("*") * 100).alias("card_payment_pct"),
    )
    tips = card.agg(
        F.avg("tip_amount").alias("avg_card_tip"),
        F.percentile_approx(F.col("tip_amount") / F.col("fare_amount") * 100,
                            0.5).alias("median_tip_pct"),
    )
    return main.crossJoin(tips).withColumn("year", F.lit(label))

comparison = average_metrics(trips_clean, "2019").unionByName(
    average_metrics(recent_clean, "2026"))

# one line per metric, one column per year, plus the change in %
metrics = [c for c in comparison.columns if c != "year"]
rows = {r["year"]: r for r in comparison.collect()}
print(f"{'metric':18s}{'2019':>14s}{'2026':>14s}{'change':>10s}")
for m in metrics:
    a, b = rows["2019"][m], rows["2026"][m]
    print(f"{m:18s}{a:14,.2f}{b:14,.2f}{(b - a) / a * 100:+9.1f}%")

metric                      2019          2026    change
trips               7,613,478.00  3,515,654.00    -53.8%
avg_passengers              1.57          1.25    -20.0%
avg_distance_mi             2.85          3.49    +22.7%
avg_duration_min           13.00         17.14    +31.9%
avg_fare                   12.37         21.08    +70.4%
avg_total_paid             15.65         29.67    +89.6%
card_payment_pct           71.56         62.35    -12.9%
avg_card_tip                2.53          4.11    +62.3%
median_tip_pct             22.22         26.61    +19.8%


In [29]:
# passenger_count is missing for a lot of trips in 2026
print("2026 trips with passenger_count = NULL:",
      recent_clean.filter(F.col("passenger_count").isNull()).count())
recent_clean.groupBy("payment_type").count().orderBy("payment_type").show()

# average fare by pick-up borough, 2019 vs 2026
recent_boro = recent_clean.join(F.broadcast(pickup_zones), on="PULocationID", how="left")
fare_2019 = trips_boro.groupBy("pickup_borough").agg(
    F.round(F.avg("fare_amount"), 2).alias("avg_fare_2019"))
fare_2026 = recent_boro.groupBy("pickup_borough").agg(
    F.round(F.avg("fare_amount"), 2).alias("avg_fare_2026"),
    F.count("*").alias("trips_2026"))
(fare_2019
 .join(fare_2026, on="pickup_borough", how="outer")
 .withColumn("change_pct",
             F.round((F.col("avg_fare_2026") / F.col("avg_fare_2019") - 1) * 100, 1))
 .orderBy(F.desc("trips_2026"))
 .show())

2026 trips with passenger_count = NULL: 994636


+------------+-------+
|payment_type|  count|
+------------+-------+
|           0| 994636|
|           1|2192041|
|           2| 292288|
|           3|   9070|
|           4|  27619|
+------------+-------+



+--------------+-------------+-------------+----------+----------+
|pickup_borough|avg_fare_2019|avg_fare_2026|trips_2026|change_pct|
+--------------+-------------+-------------+----------+----------+
|     Manhattan|        10.72|        17.53|   3008642|      63.5|
|        Queens|        35.69|        48.51|    320389|      35.9|
|      Brooklyn|        18.83|        30.88|    146129|      64.0|
|         Bronx|        26.89|        32.69|     35159|      21.6|
|       Unknown|        11.76|        20.38|      4085|      73.3|
|           N/A|        40.04|        74.47|       680|      86.0|
| Staten Island|        44.67|        41.72|       454|      -6.6|
|           EWR|         77.8|        95.09|       116|      22.2|
+--------------+-------------+-------------+----------+----------+



Yes, almost all the average metrics changed between January 2019 and January 2026:

- **Number of trips: -54 %** (3.5 million instead of 7.6 million). Yellow taxis are used much less, probably because of the competition from Uber and Lyft.
- **Average fare: +70 %** (21.08 $ instead of 12.37 $), and **total paid: +90 %** (29.67 $ instead of 15.65 $). The prices went up (inflation and the new taxi fares), and the total also includes new fees, like the Manhattan congestion fee (the new `cbd_congestion_fee` column, which did not exist in 2019).
- **Average distance: +23 %** (3.49 instead of 2.85 miles) and **average duration: +32 %** (17.1 instead of 13.0 minutes). The trips are longer and slower.
- **Average card tip: +62 %** (4.11 $ instead of 2.53 $), and the median tip percentage went from 22 % to about 27 %. I think it is because the payment screens in taxis now suggest higher percentages.
- **Average passengers: -20 %** (1.25 instead of 1.57).

The fare increase is visible in every borough except Staten Island (and there are very few trips there).

Something I have to be careful about: in 2026, about **995,000 trips** have `payment_type = 0` and no passenger count. So the average passenger count and the share of card payments in 2026 are computed on fewer trips than in 2019, and these two changes are less reliable than the others.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

### Part 3: my answers

Since I did parts 1 and 2 with the DataFrame API, I redo 3 questions here in **pure SQL** with `spark.sql()`. I chose:

1. What is the Average passanger count (part 1)
2. On average which day of the week is slowest/busiest (part 1)
3. which borough had most pickups? dropoffs? (part 2, **with a join**)

To use SQL, I first need to register my DataFrames as **temporary views**, so Spark knows them by a table name. They are the same data as before (the cleaned trips and the zone table), so I should get exactly the same results.

In [30]:
trips.createOrReplaceTempView("trips")               # raw 2019 trips
trips_clean.createOrReplaceTempView("trips_clean")   # cleaned 2019 trips
df_zones.createOrReplaceTempView("zones")            # taxi zone lookup

spark.sql("SHOW VIEWS").show()

+---------+-----------+-----------+
|namespace|   viewName|isTemporary|
+---------+-----------+-----------+
|         |      trips|       true|
|         |trips_clean|       true|
|         |      zones|       true|
+---------+-----------+-----------+



#### 1. What is the Average passanger count (in SQL)

In [31]:
spark.sql("""
    SELECT
        ROUND(AVG(passenger_count), 3)                AS avg_all_rows,
        ROUND(AVG(CASE WHEN passenger_count > 0
                       THEN passenger_count END), 3)  AS avg_excluding_0
    FROM trips
""").show()

+------------+---------------+
|avg_all_rows|avg_excluding_0|
+------------+---------------+
|       1.567|          1.591|
+------------+---------------+



I get exactly the same result as in part 1: **1.567** passengers on average, and **1.591** without the trips with 0 passengers. In SQL, the `CASE WHEN` does the same job as `F.when()` in PySpark: it returns `NULL` when the passenger count is 0, and `AVG` ignores the `NULL` values.

#### 2. On average which day of the week is slowest/busiest (in SQL)

In [32]:
spark.sql("""
    -- step 1: number of trips for each date
    WITH trips_per_day AS (
        SELECT pickup_date, COUNT(*) AS trips
        FROM trips_clean
        GROUP BY pickup_date
    )
    -- step 2: average of these daily counts for each day of the week
    SELECT
        date_format(pickup_date, 'EEEE') AS day_of_week,
        COUNT(*)                         AS days_in_month,
        SUM(trips)                       AS total_trips,
        ROUND(AVG(trips))                AS avg_trips_per_day
    FROM trips_per_day
    GROUP BY date_format(pickup_date, 'EEEE')
    ORDER BY avg_trips_per_day DESC
""").show()

+-----------+-------------+-----------+-----------------+
|day_of_week|days_in_month|total_trips|avg_trips_per_day|
+-----------+-------------+-----------+-----------------+
|     Friday|            4|    1075595|         268899.0|
|   Thursday|            5|    1342909|         268582.0|
|  Wednesday|            5|    1252125|         250425.0|
|   Saturday|            4|     999135|         249784.0|
|    Tuesday|            5|    1195774|         239155.0|
|     Monday|            4|     898052|         224513.0|
|     Sunday|            4|     849888|         212472.0|
+-----------+-------------+-----------+-----------------+



Same result as in part 1: **Friday** is the busiest day on average (268,899 trips per day) and **Sunday** the slowest (212,472 trips per day).

In SQL, I used a **CTE** (`WITH trips_per_day AS (...)`) to do the two steps of part 1: first count the trips per date, then average these counts per day of the week. Without this first step, a simple `COUNT(*)` grouped by weekday would put Thursday first, only because there are 5 Thursdays in January 2019.

#### 3. which borough had most pickups? dropoffs? (in SQL, with a join)

In [33]:
spark.sql("""
    -- pick-ups: join the pick-up zone of each trip with the zone table
    WITH pickups AS (
        SELECT z.Borough AS borough, COUNT(*) AS pickups
        FROM trips_clean t
        LEFT JOIN zones z ON t.PULocationID = z.LocationID
        GROUP BY z.Borough
    ),
    -- drop-offs: same thing with the drop-off zone
    dropoffs AS (
        SELECT z.Borough AS borough, COUNT(*) AS dropoffs
        FROM trips_clean t
        LEFT JOIN zones z ON t.DOLocationID = z.LocationID
        GROUP BY z.Borough
    )
    SELECT
        p.borough,
        p.pickups,
        d.dropoffs,
        ROUND(p.pickups * 100.0 / SUM(p.pickups) OVER (), 2) AS pickups_pct
    FROM pickups p
    FULL OUTER JOIN dropoffs d ON p.borough = d.borough
    ORDER BY p.pickups DESC
""").show()

+-------------+-------+--------+-----------+
|      borough|pickups|dropoffs|pickups_pct|
+-------------+-------+--------+-----------+
|    Manhattan|6898060| 6766655|      90.60|
|       Queens| 455302|  325428|       5.98|
|      Unknown| 151157|  140623|       1.99|
|     Brooklyn|  89229|  296966|       1.17|
|        Bronx|  17175|   56853|       0.23|
|          N/A|   2096|   14417|       0.03|
|Staten Island|    322|    2132|       0.00|
|          EWR|    137|   10404|       0.00|
+-------------+-------+--------+-----------+



Same result as in part 2: **Manhattan** has the most pick-ups (6,898,060, i.e. **90.6 %**) and the most drop-offs (6,766,655).

This query needs **two joins** with the zone table: one on `PULocationID` for the pick-ups and one on `DOLocationID` for the drop-offs, like in part 2. Then I join the two results on the borough name with a `FULL OUTER JOIN`, so that a borough with only pick-ups or only drop-offs would not be lost. For the percentage I used a window function, `SUM(p.pickups) OVER ()`, which gives the total of all the pick-ups on each line.

**What I learned from part 3:** the SQL version and the DataFrame version give exactly the same numbers. It makes sense, because Spark sends both of them to the same optimizer (Catalyst), which builds the same execution plan. So choosing SQL or PySpark is more a question of readability. I find SQL easier to read for joins and aggregations, and PySpark easier when I need to reuse variables or build columns step by step (like in part 1).

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing